# 4 · One camera, two resolutions

A calibration is valid for one image mode. When the image is **resized** by $(s_x, s_y)$, all of $f_x, f_y, c_x, c_y$ scale; when it is **cropped**, only $c_x, c_y$ shift; the distortion coefficients never change, because they live in normalized coordinates, i.e. they describe the lens, not the pixels.
`calib.geometry` implements `scale_intrinsics` (with the half-pixel convention $c' = (c + 0.5)\,s - 0.5$), `crop_intrinsics` and `fov_degrees`.

Question: can the calibration of one mode predict another mode, and what is the relation between the modes?

> The webcam sections need captures in `data/webcam/` (mine are not published because they show my face). The saved outputs show my results; every synthetic section runs as is.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from testbench import gt

PATTERN = (9, 6)                              # inner corners of the checkerboard
WEBCAM = ROOT / "data" / "webcam"             # my captures (not published: they show my face)
FIG = ROOT / "docs" / "images"                # figures used in the README
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3)

In [2]:
from calib.board import object_points
from calib.detect import find_corners
from calib.calibrate import calibrate
from calib.geometry import fov_degrees, scale_intrinsics, crop_intrinsics

def calibrate_folder(folder, square_mm, pattern="*.jpg", skip=()):
    img_pts, size = [], None
    for f in sorted(Path(folder).glob(pattern)):
        if f.name in skip:
            continue
        gray = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        size = (gray.shape[1], gray.shape[0])
        c = find_corners(gray, PATTERN)
        if c is not None:
            img_pts.append(c)
    return calibrate([object_points(*PATTERN, square_mm)] * len(img_pts), img_pts, size)

def k_row(K):
    return [K[0, 0], K[1, 1], K[0, 2], K[1, 2]]

## 4.1 Synthetic: 2400×1800 (4:3) vs 1280×720 (16:9)

In [3]:
res_full = calibrate_folder(gt.data_dir("full"), 25.0, skip=["img_10.jpg"])   # refocused view excluded
res_hd = calibrate_folder(gt.data_dir("hd"), 25.0)
(W, H), (w, h) = res_full.image_size, res_hd.image_size

K_resize = scale_intrinsics(res_full.K, w / W, h / H)                         # hypothesis A: resize only
band = W * h / w                                                              # hypothesis B: 16:9 band of
y0 = (H - band) / 2                                                           # full width, centred, then
K_crop = scale_intrinsics(crop_intrinsics(res_full.K, 0, y0), w / W, w / W)   # uniform scaling

print(pd.DataFrame([k_row(res_hd.K), k_row(K_resize), k_row(K_crop)],
                   index=["1280x720 calibrated", "A: resize only", "B: crop + resize"],
                   columns=["fx", "fy", "cx", "cy"]).round(1))

s = w / W
y0_meas = res_full.K[1, 2] - (res_hd.K[1, 2] + 0.5) / s + 0.5
print(f"\nrows used by the 16:9 mode: {y0_meas:.0f}..{y0_meas + band:.0f} (centred crop: {y0:.0f}..{y0 + band:.0f})")

                         fx      fy     cx     cy
1280x720 calibrated  1065.5  1067.8  647.2  354.4
A: resize only       1066.4   801.5  646.2  354.1
B: crop + resize     1066.4  1068.6  646.2  352.3

rows used by the 16:9 mode: 221..1571 (centred crop: 225..1575)


Only **B** matches: a plain resize would need non-square pixels ($f_y$ 802 instead of 1068). The 16:9 mode reads a centred band of 1350 sensor rows (estimated from $c_y$: 221 vs 225, within calibration noise) and scales it by 1280/2400.

## 4.2 The webcam: 1920×1080 vs 640×480

In [4]:
res_1080 = calibrate_folder(WEBCAM / "calib_1920x1080", 8.24, "*.png")
res_480 = calibrate_folder(WEBCAM / "second_mode_640x480", 8.24, "*.png")

# observed in notebook 1: 640x480 = central 1440x1080 of the 16:9 image, scaled by 640/1440
K_pred = scale_intrinsics(crop_intrinsics(res_1080.K, (1920 - 1440) / 2, 0), 640 / 1440, 640 / 1440)
K_naive = scale_intrinsics(res_1080.K, 640 / 1920, 480 / 1080)

table = pd.DataFrame([k_row(res_480.K), k_row(K_pred), k_row(K_naive)],
                     index=["640x480 calibrated", "crop + resize prediction", "resize only"],
                     columns=["fx", "fy", "cx", "cy"])
table["fx error"] = (table["fx"] / res_480.K[0, 0] - 1).map("{:+.1%}".format)
print(table.round(1))
for name, r in [("1920x1080", res_1080), ("640x480", res_480)]:
    print(f"{name}: field of view {fov_degrees(r.K, r.image_size)[0]:.1f} x {fov_degrees(r.K, r.image_size)[1]:.1f} deg")

                             fx     fy     cx     cy fx error
640x480 calibrated        547.2  549.3  301.8  246.6    +0.0%
crop + resize prediction  555.5  555.1  304.5  246.8    +1.5%
resize only               416.6  555.1  308.3  246.8   -23.9%
1920x1080: field of view 75.1 x 46.8 deg
640x480: field of view 60.6 x 47.2 deg


The crop + resize model predicts the independently calibrated 640×480 intrinsics within **≈ 1.5 %** in $f$ and ≈ 3 px in $c$; treating the mode as a plain resize gets $f_x$ wrong by **24 %**. The horizontal field of view drops from 75° to 61°: the 4:3 mode really sees less.

**Takeaways**
- A calibration belongs to one mode (resolution, crop, zoom, focus).
- If the exact relation between modes is known, $K$ can be transferred with `crop_intrinsics` + `scale_intrinsics`; distortion stays the same. Verify the transfer with a hold-out error on a few photos.
- Never assume a mode is a plain resize: check the field of view first.
- The half-pixel term in $c' = (c + 0.5)s - 0.5$ is a 0.25 px effect at 2× downscaling, negligible compared with the calibration uncertainty of $c$, but free to get right.